# Load Raw Weather Data from Bronze Layer

This step loads the raw weather data previously stored in the Bronze layer into a Spark DataFrame.

The Bronze table serves as the source for all subsequent data transformations in the Silver layer.

No transformations are applied at this stage.

In [0]:
# Define the source Bronze table
BRONZE_TABLE = "Workspace.weather_data.weather_raw"

# Load raw weather data from Bronze table
bronze_df = spark.read.table(BRONZE_TABLE)

# Display the DataFrame schema
bronze_df.printSchema()

# Display the first 5 records
display(bronze_df.limit(5))

# Count total number of records
print(f"Total nubmber of records: {bronze_df.count()}")


# Convert Measurement Time to Timestamp

This step converts the measurement_time column from string to timestamp.

Using the timestamp data type enables proper time-based filtering, sorting, and aggregation in subsequent processing stages.

All other columns remain unchanged.

In [0]:
from pyspark.sql.functions import col, to_timestamp

# Convert measurement_time column from string to timestamp
silver_df = bronze_df.withColumn(
    "measurement_time", to_timestamp(col("measurement_time"))
)

# Display the updated DataFrame schema
silver_df.printSchema()

# Display top 5 rows
display(silver_df.limit(5))

# Validate Measurement Time Conversion

This step verifies that the measurement_time column was successfully converted to timestamp.

It checks whether any null values were introduced during the conversion.

In [0]:
# Count records with null measurement_time values
null_timestamps = silver_df.filter(col("measurement_time").isNull()).count()

# Display number of records with null measurement_time
print(f"Number of records with null measurement_time: {null_timestamps}")

# Validate Null Values in City

This step checks whether the city column contains null values.

Every weather record must be associated with a valid city.
The pipeline must stop if null values are detected.

In [0]:
# Count null values in the city column
city_null_values = silver_df.filter(col("city").isNull()).count()

# Stop the pipeline if null values are detected
if city_null_values > 0:
    raise ValueError(
        "Data quality check failed: {city_null_values} detected in the city column"
    )
print("Data quality check passed: No Null values in the city column")

# Validate Empty Values in City

This step checks whether the city column contains empty strings or values consisting only of whitespace.

Every weather record must contain a non-empty city name.
The pipeline must stop if empty values are detected.

In [0]:
from pyspark.sql.functions import trim

# Count empty or whitespace-only city values
empty_city_count = silver_df.filter(trim(col("city")) == "").count()

# Stop the pipeline if empty values are detected
if empty_city_count > 0:
    raise ValueError(f"Data quality check failed: {empty_city_count} empty values found in city column")
else:
    print("Data quality check passed: No empty values found in city column")

# Check Leading and Trailing Whitespace in City Names

This step identifies city names containing leading or trailing whitespace.

The original string length is compared with the trimmed string length to detect unnecessary whitespace.

In [0]:
from pyspark.sql.functions import length

# Identify city names with leading or trailing whitespaces
whitespace_city_df = (
    silver_df.filter(length(trim(col("city"))) != length(col("city")))
    .select("city")
    .distinct()
)

# Display detected city names
display(whitespace_city_df)

# Count number of cities with leading or trailing whitespaces
whitespace_count = whitespace_city_df.count()

if whitespace_count > 0:
    print(
        f"Data quality check failed: number of distinct values with leading or trailing whitespaces in city column: {whitespace_count}"
    )
else:
    print(
        "Data quality check passed: No Leading or Trailing whitespaces found in city column"
    )

# Remove Leading and Trailing Whitespace

This step removes leading and trailing whitespace from city names when necessary.

The transformation is applied only if whitespace issues have been detected.

In [0]:
# # Check whether leading or trailing whitespace was detected
trim_applied = whitespace_count > 0

# Apply trim() only when necessary
if trim_applied:
    silver_df = silver_df.withColumn("city", trim("city"))
    print("Leading or trailing spaces removed from city column")
else:
    print("No leading or trailing spaces found in city column")

# Validate City Names After Whitespace Removal

This step verifies that no leading or trailing whitespace remains in the city column after the transformation.

The pipeline stops if any whitespace issues are still detected.

In [0]:
if trim_applied:
    # Count city values with leading or trailing whitespace
    remaining_whitespace_city_count = silver_df.filter(
        length(trim(col("city"))) != length(col("city"))
    ).count()
    if remaining_whitespace_city_count == 0:
        print("Data quality check passed: ")
        print("No leading or trailing spaces found in city column")
    else:
        raise ValueError(
            f"Data quality check failed: {remaining_whitespace_city_count} records still containing leading or trailing whitespaces in city column"
        )
else:
    print(
        "Data quality check passed: No whitespaces transformation was reqiured"
    )

# Validate Uniqueness of City and Measurement Time

This step verifies that each city has only one record for each measurement_time.

The combination of city and measurement_time must be unique.

The pipeline stops if duplicate combinations are detected.

In [0]:
# Count duplicate city and measurement_time combinations
duplicate_count = (
    silver_df.groupBy("city", "measurement_time").count().filter(col("count") > 1).count())
display(duplicate_count)

# Stop the pipeline if duplicate combinations are detected
if duplicate_count > 0:
    raise ValueError(f"Data quality check failed: {duplicate_count} city and measurement_time combinations are duplicated")
else:
    print("Data quality check passed: city and measurement_time combinations are unique")

# Validate Completeness of Latitude and Longitude

This step verifies that every weather record contains valid geographic coordinates.

The latitude and longitude columns must not contain NULL values.

The pipeline stops if NULL values are detected in either column.

In [0]:
# Count NULL values in latitude and longitude
latitude_null_count = silver_df.filter(col("latitude").isNull()).count()
longitude_null_count = silver_df.filter(col("longitude").isNull()).count()

# Stop the pipeline if NULL values are detected
if latitude_null_count > 0 or longitude_null_count > 0:
    raise ValueError(
        f"Data quality check failed: {latitude_null_count} Null values found in latitude column and {longitude_null_count} Null values found in longitude column"
    )
else:
    print("Data quality check passed: latitude and longitude values are not Null")

# Add Time Attributes

This step derives additional time attributes from the measurement_time column.

The new columns will support time-based filtering and aggregations in the Gold layer.

The original measurement_time column remains unchanged.

In [0]:
from pyspark.sql.functions import to_date, year, month, quarter, dayofmonth

# Add time-related columns
silver_df = (
    silver_df.withColumn("date", to_date(col("measurement_time")))
    .withColumn("year", year(col("measurement_time")))
    .withColumn("month", month(col("measurement_time")))
    .withColumn("quarter", quarter(col("measurement_time")))
    .withColumn("day_of_month", dayofmonth(col("measurement_time"))) 
)

# Display the updated DataFrame schema
display(silver_df.schema)

# Display the first 5 rows of the updated DataFrame
display(silver_df.limit(10))

# Validate Time Attributes

This step verifies that all derived time attributes contain valid values.

The pipeline stops if any of the new time-related columns contain NULL values.

In [0]:
# Define the derived time columns to validate
time_columns = ["date", "year", "month", "quarter", "day_of_month"]

# Count NULL values in each derived time column
for column in time_columns:
    null_count = silver_df.filter(col(column).isNull()).count()

    if null_count > 0:
        raise ValueError(f"Data quality check failed: {null_count} Null values found in {column}")
    else:
        print(f"Data quality check passed: No Null values found in {column}")

# Reorder Silver DataFrame Columns

This step rearranges the columns in the Silver DataFrame to improve readability and logical organization.

Columns are grouped by their meaning, with city and measurement_time placed at the beginning.

No data values are modified or removed.

In [0]:
# Define the desired column order
column_order = [
    "city",
    "latitude",
    "longitude",
    "measurement_time",
    "date",
    "year",
    "month",
    "quarter",
    "day_of_month",
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "wind_direction_10m",
    "shortwave_radiation"
]   

# Reorder DataFrame columns
silver_df = silver_df.select(*column_order)

# Display the updated DataFrame
display(silver_df.limit(10))

# Save Transformed Data to Silver Layer

This step saves the cleaned, validated, and organized weather data to a managed Delta table in the Silver layer.

The Silver table serves as the source for subsequent aggregations in the Gold layer.

In [0]:
# Define the Silver layer table name
SILVER_TABLE = "Workspace.weather_data.weather_silver"

# Save the transformed data as a Delta table
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)

# Confirm successful data storage
print(f"Data successfully stored in the {SILVER_TABLE}")


# Validate Silver Table

This step verifies that the transformed weather data has been successfully saved to the Silver layer.

The validation checks whether the table can be loaded and whether its column names and order match the structure defined during the transformation stage.


In [0]:
# Load the saved Silver table
silver_saved_df = spark.read.table(SILVER_TABLE)
# Validate the column names and their order
if silver_saved_df.columns != column_order:
      raise ValueError(
        "Silver table validation failed: column names or their order do not match the expected structure."
    )
else:
    print("Silver table validation passed: column names and their order match the expected structure.")

# Report total number of records
print(f"Total number of records in the Silver table: {silver_saved_df.count()}")

# Report total number of columns
print(f"Total number of columns in the Silver table: {len(silver_saved_df.columns)}")

# Confirm successful validation
print("Silver table validation completed successfully.")


In [0]:
display(silver_saved_df.schema)